<style>
table {
  margin-left: 0 !important;
  margin-right: auto !important;
}
th, td {
  text-align: left !important;
}
</style>


## 02-1 · Part 1: From a Case Description to a Formulation

**A formulation makes the choice, response, comparison rule, and requirements precise enough to evaluate any candidate.**

Lecture 01-2 identified decision variables, objectives, and constraints in five everyday cases. This lecture keeps those cases and gives each one standard mathematical notation. Part 1 introduces the common template through Case A, the digital clock.

### 1 · Use one standard template

Let \(x\) be the decision vector and let \(\mathcal X\) be its allowed domain. A system model or direct calculation produces responses \(y=\operatorname{Sim}(x)\). A scalar objective \(f\) compares feasible candidates. Inequality and equality residuals express requirements.

> $\displaystyle \underset{x\in\mathcal X}{\operatorname{minimize}}\quad f(y)$
>
> $\displaystyle \text{subject to}\quad y=\operatorname{Sim}(x),$
>
> $\displaystyle g_j(x,y)\le0\quad(j=1,\ldots,m_g),$
>
> $\displaystyle h_k(x,y)=0\quad(k=1,\ldots,m_h).$

Here, \(m_g\) and \(m_h\) are the numbers of inequality and equality constraints. Maximization can be written in the minimization template by negating the quantity to be maximized.

For a dynamic system, \(\operatorname{Sim}\) can repeat the physical state transition \(F\) and then apply the performance mapping \(G\). For the classroom, the standard-form objective \(f(y;\lambda_E)\) is the score supplied by \(H\): \(f(y;\lambda_E)=J(u;\lambda_E)\). Uppercase \(F\) remains reserved for the physical state transition.

### 2 · Formulate Case A: correct the clock

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/02-1_problem_formulation/assets/case_a_clock.png" alt="A classroom clock reading 09:59, a reference clock reading 10:00, and an adjustable time-correction control." width="570" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

Case A from 01-2: the clock correction becomes the decision \(x=[q]\). The remaining errors are the responses \(y\).

The observed clock errors are \(d=(-2,-1,-1,0)\) minutes. The stored correction \(q\) is the real choice. In standard notation,

| Formulation part | Clock case |
|:---|:---|
| Decision | \(x=[q]\) |
| Domain | \(\mathcal X=\mathbb R\) |
| Response | \(y=\operatorname{Sim}(x)=d+q\mathbf 1\) |
| Objective | \(f(y)=\sum_{i=1}^{4}y_i^2\) |
| Constraints | None, so \(m_g=m_h=0\) |

The complete formulation is

> $\displaystyle \underset{q\in\mathbb R}{\operatorname{minimize}}\quad
\sum_{i=1}^{4}(d_i+q)^2.$

The vector \(y\) contains the remaining errors. It is produced by the chosen correction and is not another decision.

The next cell defines the fixed observations, response calculation, and objective separately.

In [ ]:
import numpy as np

# Fixed observations
OBSERVED_ERRORS = np.array([-2.0, -1.0, -1.0, 0.0])


def simulate_clock(x):
    '''Return y=Sim(x): remaining clock errors after the correction.'''
    correction = float(np.asarray(x)[0])
    return OBSERVED_ERRORS + correction


def objective_function(y):
    '''Return the sum of squared remaining errors.'''
    return float(np.sum(np.asarray(y) ** 2))


def evaluate_candidate(x):
    '''Evaluate one unconstrained clock-correction candidate.'''
    decision = np.asarray(x, dtype=float)
    response = simulate_clock(decision)
    return {"x": decision, "y": response, "f": objective_function(response)}

The next calculation compares a stated 0.25-minute grid with the continuous solution.

In [ ]:
grid = np.arange(-1.0, 2.0 + 0.125, 0.25)
grid_records = [evaluate_candidate([correction]) for correction in grid]
best_grid = min(grid_records, key=lambda record: record["f"])

# For this quadratic, differentiating gives the exact continuous minimizer.
continuous_solution = -float(np.mean(OBSERVED_ERRORS))
continuous_record = evaluate_candidate([continuous_solution])

print(f"Best 0.25-minute grid candidate: q={best_grid['x'][0]:.2f}, f={best_grid['f']:.2f}")
print(f"Continuous minimizer: q={continuous_solution:.2f}, f={continuous_record['f']:.2f}")

### 3 · Classify only after the formulation is complete

| Classification axis | Clock formulation | Evidence |
|:---|:---|:---|
| Constraints | Unconstrained | No explicit bounds, inequalities, or equalities |
| Decision domain | Continuous | \(q\in\mathbb R\) |
| Objectives | Single-objective | One scalar sum is minimized |
| Function structure | Nonlinear | The objective contains squared terms |
| Response evaluation | Direct algebraic | The remaining errors are calculated directly |
| Uncertainty | Deterministic | One fixed set of observed errors is used |

The reported grid result is the best candidate on that finite grid. It is not, by itself, proof of the continuous optimum. Here the separate analytic calculation supplies the continuous minimizer.

If the domain changes to \([-1,1]\), the stored correction still changes the real clock. The bound changes only which corrections are eligible.

### Takeaway

Translate a case in a fixed order:

> **decision \(x\) and domain \(\mathcal X\) → response \(y=\operatorname{Sim}(x)\) → feasibility from \(g,h\) → comparison by \(f\) → problem classification**

Case A has \(m_g=m_h=0\), so feasibility is automatic for every \(q\in\mathbb R\). Part 2 applies the same template to a case with spending bounds and a coupled budget requirement.